# 12주 · 가위치법의 정체·선형 회귀

수치해석및실습 CE273 · 2027학년도 1학기

원본 슬라이드의 개념·예제·차시 순서에 연결한 **신규 병행 실습**. 원본 코드의 단순 복사본과 구분

**목표**: 수치 원리·도구 기능 이해 → 좋은 AI 질문·지시 → 오차·수렴·결과의 검토

**수업**: 2027-05-18 방정식의 근 (3) · 온라인 영상강의 / 2027-05-20 곡선 적합 (1) · 대면

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE273-numerical-analysis/week12/CE273_W12.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE273-numerical-analysis/week12)

슬라이드 예측 → 작은 손계산 → 실행 → 한 조건 변경 → AI 요청 수정 → 단위·오차·수렴·실패 상태 검토

셀 실행: Shift+Enter. 처음에는 위에서 아래로 실행. 공통 준비 셀 실행 후 각 절의 개별 실행도 가능. None 항목과 변경 셀은 직접 작성

## 0. 실행 환경·보조 함수

필요한 버전만 설치. 업데이트한 패키지를 이미 불러온 경우 런타임 재시작 후 처음부터 실행

보조 함수의 세부 문법 암기 불필요. 입력·처리·반환값·오차 기준·실패 조건부터 확인. 아래 실습의 호출 코드와 슬라이드 식 대조

In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.11", "matplotlib": "3.8", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, linalg, interpolate, integrate

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 자료·최소 원소 수 확인 필요")
    return x

def relative_error(truth, estimate):
    # 기준값 0에서 상대오차를 정의하지 않음. 절대오차 별도 확인
    return np.nan if truth == 0 else abs((truth-estimate)/truth)
def root_result(x, history, converged, reason):
    return {"root":float(x),"history":pd.DataFrame(history),"iterations":len(history),"converged":converged,"reason":reason}

def root_options(xtol,ftol,max_iter):
    if not np.isfinite([xtol,ftol]).all() or xtol<=0 or ftol<=0 or not isinstance(max_iter,int) or max_iter<=0:
        raise ValueError("양의 위치·잔차 허용오차, 양의 정수 최대 반복 수 필요")

def bracket_root(f,a,b,method="bisection",xtol=1e-10,ftol=1e-10,max_iter=200):
    root_options(xtol,ftol,max_iter)
    if method not in ["bisection","false_position","illinois"] or not np.isfinite([a,b]).all() or a>=b:raise ValueError("방법·a<b 확인 필요")
    fa,fb=float(f(a)),float(f(b));history=[]
    if not np.isfinite([fa,fb]).all():raise ValueError("유한한 양 끝 함수값 필요")
    if fa==0:return root_result(a,history,True,"left endpoint")
    if fb==0:return root_result(b,history,True,"right endpoint")
    if np.signbit(fa)==np.signbit(fb):raise ValueError("연속성 확인 후 부호 변화 구간 필요")
    wa,wb=fa,fb;previous=None;retained=None;x=(a+b)/2
    for k in range(max_iter):
        x=(a+b)/2 if method=="bisection" else b-wb*(b-a)/(wb-wa)
        fx=float(f(x))
        if not np.isfinite(fx):raise ValueError("반복 중 비유한 함수값")
        step=np.inf if previous is None else abs(x-previous)
        history.append({"iteration":k+1,"a":a,"b":b,"x":x,"residual":abs(fx),"step":step})
        if fx==0 or (abs(fx)<=ftol and (b-a<=2*xtol if method=="bisection" else step<=xtol)):
            return root_result(x,history,True,"position and residual")
        if np.signbit(fa)!=np.signbit(fx):
            b,fb,wb=x,fx,fx
            wa=wa/2 if method=="illinois" and retained=="a" else fa
            retained="a"
        else:
            a,fa,wa=x,fx,fx
            wb=wb/2 if method=="illinois" and retained=="b" else fb
            retained="b"
        previous=x
    return root_result(x,history,False,"maximum iterations")

def open_root(f,x0,method="newton",df=None,g=None,x1=None,xtol=1e-10,ftol=1e-10,max_iter=100):
    root_options(xtol,ftol,max_iter)
    if method not in ["newton","fixed_point","secant"] or not np.isfinite(x0):raise ValueError("방법·초기값 확인 필요")
    if method=="newton" and not callable(df):raise ValueError("도함수 필요")
    if method=="fixed_point" and not callable(g):raise ValueError("고정점 함수 필요")
    if method=="secant" and (x1 is None or not np.isfinite(x1) or x1==x0):raise ValueError("서로 다른 두 유한 초기값 필요")
    previous=float(x0);current=float(x1 if method=="secant" else x0);history=[]
    if float(f(current))==0:return root_result(current,history,True,"initial root")
    for k in range(max_iter):
        fc=float(f(current))
        if method=="fixed_point":nxt=float(g(current))
        elif method=="newton":
            derivative=float(df(current))
            if not np.isfinite(derivative) or abs(derivative)<np.finfo(float).eps:raise ValueError("0 또는 작은 도함수")
            nxt=current-fc/derivative
        else:
            fp=float(f(previous));den=fc-fp
            if not np.isfinite(den) or abs(den)<np.finfo(float).eps:raise ValueError("0 또는 작은 함수값 차이")
            nxt=current-fc*(current-previous)/den
        if not np.isfinite(nxt):raise ValueError("비유한 반복값")
        residual=abs(float(f(nxt)));step=abs(nxt-current)
        if not np.isfinite(residual):raise ValueError("비유한 잔차")
        history.append({"iteration":k+1,"x":nxt,"residual":residual,"step":step})
        if residual==0 or (residual<=ftol and step<=xtol):return root_result(nxt,history,True,"position and residual")
        previous,current=current,nxt
    return root_result(current,history,False,"maximum iterations")


## S01. 기존 Colab의 x^10−1과 수정 가위치법

**슬라이드 연결**: 2026_S1_NA_Wk11a_Root1.pptx p29 / 2026_S1_NA_Wk12a_Root3.pptx p3, 4, 5, 6, 7, 8, 9, 10

**자료 구분**: 원본 예제·수치·원리에 기반. 코드에 표시한 추가 검토는 조건 변경 실험

**먼저 예측**: 애니메이션이 먼저 끝나면 두 방법 모두 수렴했다는 뜻인가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록

In [ ]:
f=lambda x:x**10-1
standard=bracket_root(f,0,1.3,"false_position",max_iter=100)
improved=bracket_root(f,0,1.3,"illinois",max_iter=100)
for label,result in [("standard",standard),("Illinois",improved)]:
    print(label,result["root"],result["iterations"],result["converged"],result["reason"])
    plt.semilogy(result["history"].iteration,result["history"].residual,label=label)
assert improved["converged"] and np.isclose(improved["root"],1)
plt.xlabel("iteration");plt.ylabel("absolute residual");plt.legend();plt.show()
print("원본 두 함수의 tol은 잔차와 상대변화율(%)로 단위가 달랐음. 여기서는 같은 기준")

**조건 변경**: 반복 상한10/100 변경 후 수렴 여부와 기록 길이 비교

원본 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**AI 요청에 넣을 정보**: 기존 함수x^10−1·구간0~1.3·가중치/실제함수값 구분·공통 종료 기준·전체 이력 요청

**결과 검토**: 원본 mp4 자동 저장 의존과 min(history 길이) 절단 제거. 반복 상한을 성공으로 처리하지 않음

예상·실행·독립 검산·수정 지시·채택 또는 거절 이유 기록

In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## S02. 원본 8쌍의 선형 회귀와 로그 변환

**슬라이드 연결**: 2026_S1_NA_Wk12b_CurveFit1 .pptx p29, 30, 31, 41

**자료 구분**: 원본 예제·수치·원리에 기반. 코드에 표시한 추가 검토는 조건 변경 실험

**먼저 예측**: 로그 변환 후의 최소제곱은 원척도에서 같은 최적화인가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록

In [ ]:
x=np.arange(10,81,10,dtype=float);y=np.array([25,70,380,550,610,1220,830,1450],dtype=float)
X=np.column_stack([np.ones(len(x)),x]);beta,residuals,rank,singular=np.linalg.lstsq(X,y,rcond=None)
r=y-X@beta
print("절편·기울기:",beta,"SSE:",r@r,"rank:",rank)
assert np.allclose(X.T@r,0,atol=1e-8)
# 로그 회귀는 원래 척도와 다른 잔차를 최소화하는 모형
logX=np.column_stack([np.ones(len(x)),np.log10(x)])
logbeta=np.linalg.lstsq(logX,np.log10(y),rcond=None)[0]
print("log10 모형:",logbeta,"원척도 계수:",10**logbeta[0])
assert np.isclose(beta[1],19.470238095238095)
plt.scatter(x,y);plt.plot(x,X@beta);plt.xlabel("x");plt.ylabel("y");plt.show()

**조건 변경**: y 한 점 변경 후 계수·잔차·로그 모형 비교

원본 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**AI 요청에 넣을 정보**: 원본8쌍·절편·최소제곱 척도·단위·rank·잔차·해석 범위 명시

**결과 검토**: 공학통계학의 같은 8쌍 예제와 수치 대조. 회귀만으로 인과 단정 불가

예상·실행·독립 검산·수정 지시·채택 또는 거절 이유 기록

In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## 주차 확인

같은 종료 기준·같은 자료·같은 목적함수끼리 비교

입력 항목:

- `기존_Colab_정확근`: 해당 계산값·항 수·오차 기준의 수치
- `원본_선형회귀_기울기`: 해당 계산값·항 수·오차 기준의 수치
- `로그회귀와_원척도회귀의_목적함수는_항상_같음`: 판단 · Python `True`/`False`

공개 계산·개념 확인용 항목. 값만 복사하면 학습 완료로 판단 불가. 가정·적용 범위·검산 근거는 아래 서술과 함께 확인

In [ ]:
answers = {'기존_Colab_정확근': None, '원본_선형회귀_기울기': None, '로그회귀와_원척도회귀의_목적함수는_항상_같음': None}


In [ ]:
def check_answers(values):
    expected = {'기존_Colab_정확근': 1.0, '원본_선형회귀_기울기': 19.470238095238095, '로그회귀와_원척도회귀의_목적함수는_항상_같음': False}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## AI 요청·검토 기록

1. 문제·모형·입력·단위·자료 출처
2. 최초 요청: 목적·조건·알고리즘·출력·검증 기준
3. 실행 전 예측·작은 손계산의 근거
4. 의심한 출력·오차·수렴·실패 조건과 수정 지시
5. 채택·수정·거절한 내용, 독립 검산·한계
6. AI 없이 설명 가능한 원리 한 가지

글자 수 자동 채점 없음. 실행 성공·작은 갱신량·작은 잔차와 문제 해결의 타당성 구분

In [ ]:
review_log = {
    "문제·모형·자료·단위": None,
    "최초 AI 요청": None,
    "예측·손계산": None,
    "의심한 부분·수정 지시": None,
    "검산·최종 판단·한계": None,
    "내 설명": None,
}
